# 1. Import the necessary libraries

In [2]:
# Adds the parent directory to sys.path
import sys
sys.path.append('../')

from problems.benchmark_problems import get_problem
from problems.microgrid_function import microgrid_function
from runners import run_mesh, run_mesh_old, run_nsga2

from joblib import Parallel, delayed

import contextlib
import io
import numpy as np

# 2. Define the general configuration of the experiments

In [ ]:
#################### CUSTOMIZABLE ####################

# Number of processes to execute the experiments in parallel
workers = 20

# Objective function configuration (function name, number of objectives, number of decision variables)
experiments = [('Lead_Acid', 3, 3, None), ('Li-ion', 3, 3, None), ('ZEBRA', 3, 3, None), ('NaS', 3, 3, None), ('NiCd', 3, 3, None), ('NiMH', 3, 3, None), ('RFV', 3, 3, None), ('ZnBr', 3, 3, None)]

# Execution configuration
num_runs = 30 # Number of runs
max_fitness_eval = 10000 # Maximum fitness evaluations (not used if it is None)
population_size = 100 # Population size
random_state = None # Set a seed for random numbers (not used if it is None)

results_folder = f'../results' # Folder to store the results
tuning_folder = f'../hyperparams' # Folder to get the tuned parameters
######################################################

# 3 Define auxiliar functions

In [4]:
# Function to get the fitness function configuration
def get_exp_problem(func_name, objective_dim, position_dim, wfg_k):
	select_bat = {'Lead_Acid': 0, 'Li-ion': 1, 'ZEBRA': 2, 'NaS': 3, 'NiCd': 4, 'NiMH': 5, 'RFV': 6, 'ZnBr': 7}
	# Microgrid problem
	if func_name in select_bat:
		load = np.genfromtxt('../seasonal_data/load.txt')
		temperature = np.genfromtxt('../seasonal_data/temperature.txt')
		solar_data = np.genfromtxt('../seasonal_data/irradiance.txt')
		wind_data = np.genfromtxt('../seasonal_data/wind.txt')
		func = lambda args: microgrid_function(args[0], args[1], args[2], select_bat[func_name], load, temperature, solar_data, wind_data)
		position_min_value = np.array([0.0, 0.0, 0.0])
		position_max_value = np.array([1000.0, 1000.0, 1000.0])
	# Benchmark problems
	else:
		func, position_min_value, position_max_value = get_problem(func_name, n_var=position_dim, n_obj=objective_dim, wfg_k=wfg_k)
	return func, objective_dim, position_dim, position_min_value, position_max_value

# Function to capture errors during parallel execution without stopping the other processes
def safe_run(run_function, params):
	try:
		f = io.StringIO()
		with contextlib.redirect_stdout(f):
			status = run_function(*params)
		return status, f.getvalue()
	except Exception as e:
		return f'Error: {str(e)}'

# 4. Run experiments with the algorithms (in parallel)

## 4.1 Run MESH

In [ ]:
#################### CUSTOMIZABLE ####################

# MESH fixed parameters
mesh_memory_size = population_size # Maximum number of particles in memory
mesh_global_best_type = [0,1] # 0 -> Sigma method (G1) | 1 -> Sigma method in fronts (G2)
mesh_dm_pool_type = [0,1,2] # 0 -> Sampling from memory (S1) | 1 -> Sampling from population (S2) | 2 -> Sampling from memory and population (S3)
mesh_differential_evolution_type = [0,1,2,3,4] # 0 -> DE\rand\1\Bin (D1) | 1 -> DE\rand\2\Bin (D2) | 2 -> DE/Best/1/Bin (D3) | 3 -> DE/Current-to-best/1/Bin (D4) | 4 -> DE/Current-to-rand/1/Bin (D5)


# MESH tunable parameters
# OBS: The function "run_mesh" and "run_mesh_old" will select automatically the tuned parameters if they exists ("hyperparams" folder generated by "fine_tuning.ipynb")
mesh_communication_probability = 0.8 # Communication probability
mesh_mutation_rate = 0.9 # Mutation rate
mesh_personal_guide_array_size = 1 # Number of personal guides
######################################################

In [6]:
# Set the list of parameters
params_list = [
  	[(F'MESH_G{gb_type+1}S{pool_type+1}D{de_type+1}_{exp[0]}_{exp[1]}_{exp[2]}', results_folder, tuning_folder, num_runs, max_fitness_eval, population_size, random_state),
	 get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
	 (mesh_memory_size, gb_type, pool_type, de_type),
	 (mesh_communication_probability, mesh_mutation_rate, mesh_personal_guide_array_size)]
     for exp in experiments
     for gb_type in mesh_global_best_type
	 for pool_type in mesh_dm_pool_type
	 for de_type in mesh_differential_evolution_type
]

# Execute in parallel
Parallel(n_jobs=workers)(delayed(safe_run)(run_mesh, params) for params in params_list)

[('MESH_G1S1D1_Lead_Acid_3_3 with tunable parameters (0.3870686234876419, 0.045102657197506835, 3) was successfully executed!',
  ''),
 ('MESH_G1S1D1_Li-ion_3_3 with tunable parameters (0.8, 0.9, 1) was successfully executed!',
  ''),
 ('MESH_G1S1D1_ZEBRA_3_3 with tunable parameters (0.8, 0.9, 1) was successfully executed!',
  ''),
 ('MESH_G1S1D1_NaS_3_3 with tunable parameters (0.8, 0.9, 1) was successfully executed!',
  ''),
 ('MESH_G1S1D1_NiCd_3_3 with tunable parameters (0.8, 0.9, 1) was successfully executed!',
  ''),
 ('MESH_G1S1D1_NiMH_3_3 with tunable parameters (0.8, 0.9, 1) was successfully executed!',
  ''),
 ('MESH_G1S1D1_RFV_3_3 with tunable parameters (0.8, 0.9, 1) was successfully executed!',
  ''),
 ('MESH_G1S1D1_ZnBr_3_3 with tunable parameters (0.8, 0.9, 1) was successfully executed!',
  '')]

### Run MESH (previous implementation)

In [ ]:
# Set the list of parameters
params_list = [
  	[(F'OLD_MESH_G{gb_type+1}S{pool_type+1}D{de_type+1}_{exp[0]}_{exp[1]}_{exp[2]}', results_folder, tuning_folder, num_runs, max_fitness_eval, population_size, random_state),
	 get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
	 (mesh_memory_size, gb_type, pool_type, de_type),
	 (mesh_communication_probability, mesh_mutation_rate, mesh_personal_guide_array_size)]
     for exp in experiments
     for gb_type in mesh_global_best_type
	 for pool_type in mesh_dm_pool_type
	 for de_type in mesh_differential_evolution_type
]

# Execute in parallel
Parallel(n_jobs=workers)(delayed(safe_run)(run_mesh_old, params) for params in params_list)

# 4.2 Run NSGA-II

In [ ]:
#################### CUSTOMIZABLE ####################

# NSGA2 tunable parameters
# OBS: The function "run_nsga2" will select automatically the tuned parameters if they exists ("hyperparams" folder generated by "fine_tuning.ipynb")
nsga2_recombination_probability = 0.45
nsga2_eta_recombination = 19
nsga2_mutation_probability = 0.06
nsga2_eta_mutation = 14.7
######################################################

# Set the list of parameters
params_list = [
  	[(F'NSGA2_{exp[0]}_{exp[1]}_{exp[2]}', results_folder, tuning_folder, num_runs, max_fitness_eval, population_size, random_state),
	 get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
	 (),
	 (nsga2_recombination_probability, nsga2_eta_recombination, nsga2_mutation_probability, nsga2_eta_mutation)]
     for exp in experiments
]

# Execute in parallel
Parallel(n_jobs=workers)(delayed(safe_run)(run_nsga2, params) for params in params_list)